# Quick test: does a deadlier chute (RACE3) change the MACURA - MBPO gap?

**RACE3 = RACE2 with one change:** dropping the chute too fast now costs up to **50%** of the lift (RACE2: 35%).
Where the lift loss starts (sinking faster than 1.2 m/s while upright and pushing) is unchanged, and so are the
course, reward, crash cost, wind, package, stabiliser and the whole protocol (the RACE2 paper protocol).

Why (hand-written autopilot, 30 scenarios, discounted return = what the agents optimise):

| autopilot at 3 m/s | RACE2 (35%) | RACE3 (50%) |
|---|---|---|
| safe: chute at 1.0 m/s | 0% crash, 205 | 0% crash, 205 |
| moderate dive: chute at 1.5 m/s | 7% crash, **213 (pays more than safe)** | 57% crash, 168 |
| dive: chute at 2.0 m/s | 27% crash, 202 | 77% crash, 149 |
| random warm-up steps losing lift | 1.0% | 0.9% (the trap stays new to the world model) |

In RACE2 diving the chute was not punished, so avoiding it could not help MACURA. In RACE3 it is.

**This run:** seed 0, **10,000 steps per algorithm = 5,000 random warm-up + 5,000 learning** (5,000 steps would be
warm-up only, no learning), MACURA, MBPO and SAC, first on RACE3, then the same on RACE2 for the comparison.
About 45-60 min on a Kaggle GPU.

**What it can and cannot show:** it checks that RACE3 runs and gives a first look. After only 5,000 learning steps
the policies rarely reach the chute's danger zone yet (in the RACE2 pilot the fastest descents stayed below 1.2 m/s
at this point), and one evaluation of 20 flights varies by about +-100-200. So a gap difference is only meaningful if
the policies actually lost lift (see "trap" below) **and** it is large. For a real answer: 30-50k steps, several seeds.

## 1. Settings and code

In [ ]:
import os, sys, time, subprocess, importlib

SEED   = 0                        # one seed
STEPS  = 10000                    # per algorithm: 5,000 random warm-up + 5,000 learning (5000 = warm-up only)
ALGOS  = "macura mbpo sac"        # the three compared (M2AC: add "m2ac")
TASKS  = ["race3", "race2"]       # race3 = the new task; race2 = the same run on the old task (the comparison)
VIDEOS = False                    # True: MACURA and MBPO fly 3 test scenarios of race3 side by side (~5 min)
REPO, BRANCH = "silvergjeka22/macura-drone", "race3"
ROOT = "/kaggle/working/macura-drone"
OUT  = "/kaggle/working/runs"     # one folder per task: runs/race3, runs/race2

os.environ.update({"MACURA_TASK": TASKS[0], "MACURA_STEPS": str(STEPS), "MACURA_SEEDS": str(SEED),
                   "MACURA_ALGOS": ALGOS, "MACURA_OUTPUT_ROOT": OUT})

token = os.environ.get("GITHUB_TOKEN")
if not token:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    f"https://{token}@github.com/{REPO}.git", ROOT], check=True)
sys.path.insert(0, ROOT)

In [ ]:
from src.bootstrap import setup
setup(root=ROOT)

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, Video, display
import src.config.config as config
from src.training.seed import set_seed
from src.training.train import train_algo
from src.analysis import results as R
from src.viz import video as V

RUNS, CFGS = {}, {}


def run_task(task):
    # the config is read when the module is (re)loaded, so each task gets its own settings; each task writes to
    # its own folder (a finished run is reloaded, not retrained)
    os.environ["MACURA_TASK"] = task
    cfg = importlib.reload(config)
    CFGS[task] = cfg.CFG
    print(f"=== {task}: max lift loss {cfg.ENV['vrs_loss']:.0%} | {cfg.TOTAL_ENV_STEPS} steps | seed {SEED} | "
          f"warm-up {cfg.WARMUP_RANDOM_STEPS} | {' '.join(cfg.ALGORITHMS)}")
    runs = []
    for algo in cfg.ALGORITHMS:
        t0 = time.time()
        set_seed(SEED)
        runs += train_algo(algo, cfg.CFG, f"{OUT}/{task}")
        print(f"--- {task} {algo} done in {(time.time() - t0) / 60:.1f} min")
    RUNS[task] = runs
    return runs

## 2. RACE3 (the new task)

In [ ]:
run_task("race3");

## 3. RACE2 (same seed, same settings, the old chute)

In [ ]:
if "race2" in TASKS:
    run_task("race2");

## 4. Results

- **Scores per task**: average return while learning and drones broken after the warm-up (the main scores), then the
  best checkpoint's test on 30 fresh scenarios.
- **The gap**: MACURA minus MBPO on each score, RACE3 next to RACE2.
- **The trap**: did the policies reach the danger zone? Fastest descent (lift loss starts at 1.2 m/s), the share of
  evaluation time spent losing lift, and MACURA's trust in fast descents vs normal flight. If nobody lost lift, the two
  tasks were practically the same so far, and any gap difference is noise.

In [ ]:
for task, runs in RUNS.items():
    display(Markdown(f"### {task}\n\n" + R.summary_markdown(runs)))
    print(R.trust_summary(runs))

SCORES = [("avg_return", "avg return while learning", "{:+.0f}"), ("broken", "drones broken after warm-up", "{:+.0f}"),
          ("test_return", "final test return", "{:+.0f}"), ("test_crash", "final test crash rate", "{:+.0%}"),
          ("test_success", "success, % of a lap (test)", "{:+.0%}")]
gap = ["| MACURA - MBPO | " + " | ".join(RUNS) + " |", "|---|" + "---|" * len(RUNS)]
for k, label, fmt in SCORES:
    cells = []
    for task, runs in RUNS.items():
        s = {r["algo"]: R.run_scores(r) for r in runs}
        cells.append(fmt.format(s["macura"][k] - s["mbpo"][k]) if {"macura", "mbpo"} <= set(s) else "-")
    gap.append(f"| {label} | " + " | ".join(cells) + " |")
display(Markdown("### The gap (positive return / success = MACURA ahead; negative broken / crash = MACURA safer)\n\n"
                 + "\n".join(gap)))

trap = ["| task | algorithm | fastest descent in the evaluations (m/s) | time losing lift (evaluations) | "
        "test: fastest descent | test: time losing lift | MACURA trusts fast / normal |", "|---|---|---|---|---|---|---|"]
reached = False
for task, runs in RUNS.items():
    for r in runs:
        fe = r["final_eval"]
        sink = max(r["eval_max_sink"]) if r["eval_max_sink"] else float("nan")
        reached |= sink > 1.2 or fe.get("eval_max_sink", 0) > 1.2
        tr = "-"
        if r["algo"] == "macura" and r.get("trust_fast"):
            f = np.nanmean([v for _, v in r["trust_fast"]]); n = np.nanmean([v for _, v in r["trust_slow"]])
            tr = f"{f:.0%} / {n:.0%}"
        trap.append(f"| {task} | {R.NAMES[r['algo']]} | {sink:.2f} | {100 * np.mean(r['eval_liftloss']):.1f}% | "
                    f"{fe.get('eval_max_sink', float('nan')):.2f} | {100 * fe.get('eval_liftloss', 0):.1f}% | {tr} |")
display(Markdown("### The trap\n\n" + "\n".join(trap)))
print("A policy reached the danger zone (> 1.2 m/s):" if reached else
      "Nobody reached the danger zone (> 1.2 m/s) yet: RACE3 and RACE2 were practically the same task in this run, "
      "so a difference in the gap is noise, not the deadlier chute.")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
for task, ls in (("race3", "-"), ("race2", "--")):
    for r in RUNS.get(task, []):
        c = {"macura": "#1f77b4", "mbpo": "#d62728", "m2ac": "#2ca02c", "sac": "#7f7f7f"}[r["algo"]]
        ax[0].plot(r["steps"], r["eval_return"], ls, color=c, lw=2, label=f"{R.NAMES[r['algo']]} {task}")
        ax[1].plot(r["steps"], r["train_crashes"], ls, color=c, lw=2, label=f"{R.NAMES[r['algo']]} {task}")
ax[0].set_ylabel("evaluation return (20 flights)"); ax[0].set_title("Return while learning (solid = race3, dashed = race2)")
ax[1].set_ylabel("real crashes so far"); ax[1].set_title("Drones broken while learning")
for a in ax:
    a.set_xlabel("real environment steps"); a.grid(alpha=0.3); a.legend(fontsize=8)
fig.tight_layout(); fig.savefig(f"{OUT}/race3_vs_race2.png", dpi=120); plt.show()

## 5. Optional: watch MACURA and MBPO on RACE3 (set VIDEOS = True)

In [ ]:
if VIDEOS and RUNS.get("race3"):
    best = {R.NAMES[r["algo"]]: r for r in RUNS["race3"] if r["algo"] in ("macura", "mbpo")}
    trust = {k: {"ensemble": R.ensemble_checkpoint(r), "kappa": R.kappa_at(r)}
             for k, r in best.items() if r["algo"] == "macura" and R.ensemble_checkpoint(r)}
    path = V.record_race(CFGS["race3"], {k: R.checkpoint(r) for k, r in best.items()}, f"{OUT}/race3/videos/race.mp4",
                         seeds=(1000, 1001, 1002), trust=trust, frame_step=3, fps=17)
    if path:
        display(Video(path, embed=True, width=960))

## 6. Outputs
Everything is in `/kaggle/working/runs/race3` and `/kaggle/working/runs/race2` (logs, checkpoints) plus
`race3_vs_race2.png`. Download with `./run.sh get`. For a longer run change `STEPS` (e.g. 30000) in section 1.